In [13]:
#!/usr/bin/env python3
# optimize_catalog_size.py
# Encuentra los parámetros óptimos para obtener ~1500 objetos de calidad

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def load_data():
    """Carga los datos ya corregidos por extinción."""
    input_file = "../anac_data/Results_Corrected/all_fields_photometry_COMPLETE_extinction.csv"
    df = pd.read_csv(input_file)
    
    print(f"📊 Datos originales cargados: {len(df)} entradas")
    
    # Eliminar duplicados exactos
    initial_count = len(df)
    df_unique = df.drop_duplicates(subset=['RAJ2000', 'DEJ2000'], keep='first')
    duplicates_removed = initial_count - len(df_unique)
    print(f"📊 Duplicados exactos eliminados: {duplicates_removed}")
    print(f"📊 Objetos únicos después de eliminar duplicados: {len(df_unique)}")
    
    # Filtro básico de probabilidad
    if 'Prob' in df_unique.columns:
        df_filtered = df_unique[df_unique['Prob'] >= 0.7]
        print(f"📊 Objetos con Prob≥0.7: {len(df_filtered)}")
    else:
        df_filtered = df_unique
    
    return df_filtered

def count_quality_objects(df, params, apertura=3):
    """
    Cuenta objetos que pasan los criterios con parámetros dados.
    """
    # Asegurar que trabajamos con objetos únicos
    df = df.drop_duplicates(subset=['RAJ2000', 'DEJ2000'])
    
    # Desempaquetar parámetros
    min_good_filters = params['min_good_filters']
    error_thresholds = params['error_thresholds']
    snr_thresholds = params.get('snr_thresholds', {})
    
    # Inicializar contador
    df['good_filters_count'] = 0
    
    # Verificar cada filtro S-PLUS
    splus_filters = ['F378', 'F395', 'F410', 'F430', 'F515', 'F660', 'F861']
    
    for filt in splus_filters:
        conditions = pd.Series(True, index=df.index)
        
        # Error
        err_col = f'MAGERR_{filt}_{apertura}_corr'
        if err_col in df.columns:
            conditions &= (df[err_col] < error_thresholds.get(filt, 0.5))
            conditions &= (df[err_col] > 0)
        
        # SNR (opcional)
        snr_col = f'SNR_{filt}_{apertura}'
        if snr_col in df.columns and filt in snr_thresholds:
            conditions &= (df[snr_col] >= snr_thresholds[filt])
        
        # Flujo válido
        flux_col = f'FLUX_{filt}_{apertura}_corr'
        if flux_col in df.columns:
            conditions &= df[flux_col].notna()
            conditions &= (df[flux_col] > 0)
        
        # Contar
        df.loc[conditions, 'good_filters_count'] += 1
    
    # Aplicar corte final
    final_mask = df['good_filters_count'] >= min_good_filters
    result_df = df[final_mask].copy()
    
    # Verificar unicidad final
    unique_count = len(result_df.drop_duplicates(subset=['RAJ2000', 'DEJ2000']))
    if len(result_df) != unique_count:
        print(f"   ⚠️  Se encontraron {len(result_df) - unique_count} duplicados en resultados, eliminando...")
        result_df = result_df.drop_duplicates(subset=['RAJ2000', 'DEJ2000'])
    
    return result_df

def optimize_parameters(target_objects=1500):
    """
    Encuentra combinaciones de parámetros que den ~1500 objetos.
    """
    df = load_data()
    
    print(f"\n🎯 OBJETIVO: Encontrar parámetros para obtener ~{target_objects} objetos")
    print(f"📊 Partiendo de {len(df)} objetos únicos con Prob≥0.7")
    print("="*60)
    
    # Estrategias diferentes
    strategies = [
        # ESTRATEGIA A: Moderada (4 filtros buenos, umbrales razonables)
        {
            'name': 'A - Balanceada',
            'min_good_filters': 4,
            'error_thresholds': {
                'F378': 0.7, 'F395': 0.6, 'F410': 0.4, 'F430': 0.4,
                'F515': 0.3, 'F660': 0.2, 'F861': 0.3
            },
            'snr_thresholds': {
                'F378': 2.0, 'F395': 2.0, 'F410': 3.0, 'F430': 3.0,
                'F515': 4.0, 'F660': 5.0, 'F861': 4.0
            }
        },
        
        # ESTRATEGIA B: Permisiva (3 filtros buenos, umbrales generosos)
        {
            'name': 'B - Permisiva',
            'min_good_filters': 3,
            'error_thresholds': {
                'F378': 0.8, 'F395': 0.7, 'F410': 0.5, 'F430': 0.5,
                'F515': 0.4, 'F660': 0.3, 'F861': 0.4
            },
            'snr_thresholds': {
                'F378': 1.5, 'F395': 1.5, 'F410': 2.0, 'F430': 2.0,
                'F515': 3.0, 'F660': 4.0, 'F861': 3.0
            }
        },
        
        # ESTRATEGIA C: Focalizada en filtros clave
        {
            'name': 'C - Filtros clave',
            'min_good_filters': 3,
            'error_thresholds': {
                'F378': 1.0, 'F395': 1.0,
                'F410': 0.4, 'F430': 0.4,
                'F515': 0.3, 'F660': 0.2,
                'F861': 0.4
            },
            'required_filters': ['F410', 'F430', 'F515', 'F660'],
            'snr_thresholds': {
                'F378': 1.0, 'F395': 1.0, 'F410': 3.0, 'F430': 3.0,
                'F515': 4.0, 'F660': 5.0, 'F861': 3.0
            }
        },
        
        # ESTRATEGIA D: Sin F378/F395
        {
            'name': 'D - Sin filtros más difíciles',
            'min_good_filters': 3,
            'error_thresholds': {
                'F410': 0.4, 'F430': 0.4,
                'F515': 0.3, 'F660': 0.2, 'F861': 0.3
            },
            'excluded_filters': ['F378', 'F395'],
            'snr_thresholds': {
                'F410': 3.0, 'F430': 3.0, 'F515': 4.0, 'F660': 5.0, 'F861': 4.0
            }
        }
    ]
    
    # Probar cada estrategia
    results = []
    for strategy in strategies:
        print(f"\n🔍 Probando estrategia: {strategy['name']}")
        
        # Aplicar criterios de calidad
        result_df = count_quality_objects(df.copy(), strategy, apertura=3)
        
        # Condición adicional para estrategia C: requerir filtros clave
        if 'required_filters' in strategy:
            temp_df = result_df.copy()
            for filt in strategy['required_filters']:
                err_col = f'MAGERR_{filt}_3_corr'
                if err_col in temp_df.columns:
                    temp_df = temp_df[temp_df[err_col] < strategy['error_thresholds'][filt]]
            result_df = temp_df
        
        n_objects = len(result_df)
        
        results.append({
            'strategy': strategy['name'],
            'n_objects': n_objects,
            'params': strategy,
            'df': result_df
        })
        
        print(f"   • Objetos obtenidos: {n_objects}")
        print(f"   • Tasa de retención: {n_objects/len(df)*100:.1f}%")
        
        # Mostrar errores medianos si hay objetos
        if n_objects > 0:
            print("   • Errores medianos en muestra:")
            for filt in ['F378', 'F395', 'F410', 'F430', 'F515', 'F660', 'F861']:
                err_col = f'MAGERR_{filt}_3_corr'
                if err_col in result_df.columns and len(result_df) > 0:
                    median_err = result_df[err_col].median()
                    print(f"     {filt}: {median_err:.3f} mag")
    
    # Encontrar la mejor estrategia
    best_idx = None
    best_diff = float('inf')
    
    for i, res in enumerate(results):
        diff = abs(res['n_objects'] - target_objects)
        if diff < best_diff:
            best_diff = diff
            best_idx = i
    
    if best_idx is not None:
        best = results[best_idx]
        print(f"\n🎯 MEJOR ESTRATEGIA: {best['strategy']}")
        print(f"   • Objetos: {best['n_objects']} (objetivo: {target_objects})")
        print(f"   • Diferencia: {abs(best['n_objects'] - target_objects)} objetos")
        
        # Guardar el mejor resultado
        output_file = f"../anac_data/Results_Corrected/all_fields_photometry_OPTIMIZED_{best['n_objects']}objects.csv"
        best['df'].to_csv(output_file, index=False)
        print(f"💾 Guardado: {output_file}")
        
        return best['df'], best['params'], best['n_objects']
    
    return None, None, 0

# Las funciones plot_optimization_results y generate_paper_text permanecen igual...

def generate_paper_text(df_optimized, params, n_objects, initial_unique_count):
    """Genera texto listo para la sección de métodos del paper."""
    print("\n" + "="*70)
    print("📝 TEXTO PARA LA SECCIÓN DE MÉTODOS DE TU PAPER")
    print("="*70)
    
    # Calcular estadísticas
    median_errors = {}
    for filt in ['F378', 'F395', 'F410', 'F430', 'F515', 'F660', 'F861']:
        err_col = f'MAGERR_{filt}_3_corr'
        if err_col in df_optimized.columns:
            median_errors[filt] = df_optimized[err_col].median()
    
    if 'Prob' in df_optimized.columns:
        prob_median = df_optimized['Prob'].median()
    else:
        prob_median = 'N/A'
    
    # Calcular porcentaje de retención
    retention_rate = (n_objects / initial_unique_count) * 100
    
    paper_text = f"""
SELECTION OF HIGH-QUALITY GLOBULAR CLUSTER CANDIDATES

From the initial catalog of {initial_unique_count} unique GC candidates around NGC 5128 
with probability P ≥ 0.7, we applied additional photometric quality criteria to 
select sources suitable for spectral energy distribution (SED) fitting with CIGALE.

Our selection process consisted of:

1. Spatial uniqueness: We removed duplicate entries based on celestial coordinates 
   (RA, DEC), retaining only unique sources.

2. Probability threshold: We selected sources with probability of being GCs P ≥ 0.7, 
   based on previous classification efforts.

3. Photometric quality control: We applied filter-specific photometric quality criteria:
   
   • We required sources to have at least {params.get('min_good_filters', 'N/A')} of the 7 
     S-PLUS narrow-band filters meeting quality thresholds.
   
   • Filter-specific error thresholds: {params['error_thresholds'].get('F378', 'N/A'):.1f} mag for F378, 
     {params['error_thresholds'].get('F395', 'N/A'):.1f} mag for F395, {params['error_thresholds'].get('F410', 'N/A'):.1f} mag for F410, 
     {params['error_thresholds'].get('F430', 'N/A'):.1f} mag for F430, {params['error_thresholds'].get('F515', 'N/A'):.1f} mag for F515,
     {params['error_thresholds'].get('F660', 'N/A'):.1f} mag for F660, and {params['error_thresholds'].get('F861', 'N/A'):.1f} mag for F861.
   
   • Signal-to-noise ratios were required to be ≥2-5 depending on the filter.

The final high-quality sample consists of {n_objects} GC candidates, representing 
{retention_rate:.1f}% of the initial probability-selected sample. These objects have 
median photometric errors ranging from {median_errors.get('F378', 0):.3f} mag in F378 to 
{median_errors.get('F660', 0):.3f} mag in F660, and a median GC probability of {prob_median:.2f}. 

This sample provides robust statistical power for our comparative analysis of 
S-PLUS-only versus S-PLUS+DECam photometric fitting while ensuring high data quality 
for reliable SED fitting results.
"""
    
    print(paper_text)
    
    # Guardar también en archivo
    with open('methodology_section.txt', 'w') as f:
        f.write(paper_text)
    print("   ✅ Texto también guardado en: methodology_section.txt")

def main():
    """Función principal de optimización."""
    print("="*70)
    print("OPTIMIZACIÓN DE PARÁMETROS PARA OBTENER ~1500 OBJETOS ÚNICOS")
    print("="*70)
    
    # Cargar datos originales para comparación
    input_file = "../anac_data/Results_Corrected/all_fields_photometry_COMPLETE_extinction.csv"
    df_full = pd.read_csv(input_file)
    
    print(f"\n📊 ANÁLISIS COMPLETO DE DATOS:")
    print(f"   • Total de entradas en archivo: {len(df_full)}")
    
    # Eliminar duplicados exactos
    df_unique = df_full.drop_duplicates(subset=['RAJ2000', 'DEJ2000'])
    print(f"   • Objetos únicos (sin duplicados): {len(df_unique)}")
    
    # Aplicar filtro de probabilidad básico
    if 'Prob' in df_unique.columns:
        df_basic = df_unique[df_unique['Prob'] >= 0.7].copy()
        print(f"   • Objetos únicos con Prob≥0.7: {len(df_basic)}")
    else:
        df_basic = df_unique.copy()
    
    initial_unique_count = len(df_basic)
    print(f"\n📊 BASE PARA OPTIMIZACIÓN: {initial_unique_count} objetos únicos con Prob≥0.7")
    
    # Optimizar para obtener ~1500 objetos
    target = 1500
    df_optimized, best_params, n_optimized = optimize_parameters(target_objects=target)
    
    if df_optimized is not None:
        # Generar gráficos (necesitarías implementar plot_optimization_results)
        # plot_optimization_results(df_optimized, initial_unique_count, n_optimized, best_params)
        
        # Generar texto para paper
        generate_paper_text(df_optimized, best_params, n_optimized, initial_unique_count)
        
        print("\n" + "="*70)
        print("✅ OPTIMIZACIÓN COMPLETADA")
        print("="*70)
        print(f"\n📊 RESUMEN FINAL:")
        print(f"   • Objetivos únicos iniciales (Prob≥0.7): {initial_unique_count}")
        print(f"   • Objetivo deseado: ~{target} objetos")
        print(f"   • Objetos obtenidos: {n_optimized} objetos únicos")
        print(f"   • Diferencia: {abs(n_optimized - target)} objetos")
        print(f"   • Tasa de retención: {n_optimized/initial_unique_count*100:.1f}%")
        
        if n_optimized < target * 0.8:
            print(f"\n⚠️  ADVERTENCIA: Menos del 80% del objetivo alcanzado.")
            print(f"   Posibles ajustes:")
            print(f"   1. Reducir umbral de probabilidad a 0.6")
            print(f"   2. Requerir solo 2 filtros buenos en lugar de {best_params.get('min_good_filters', 'N/A')}")
            print(f"   3. Aumentar todos los umbrales de error en 0.1-0.2 mag")
        
        print(f"\n📁 Archivo optimizado guardado con {n_optimized} objetos únicos")
        print(f"   Listo para análisis con CIGALE")
    
    else:
        print("\n❌ No se pudo encontrar combinación adecuada de parámetros")

if __name__ == "__main__":
    main()

OPTIMIZACIÓN DE PARÁMETROS PARA OBTENER ~1500 OBJETOS ÚNICOS

📊 ANÁLISIS COMPLETO DE DATOS:
   • Total de entradas en archivo: 5769
   • Objetos únicos (sin duplicados): 3209
   • Objetos únicos con Prob≥0.7: 2717

📊 BASE PARA OPTIMIZACIÓN: 2717 objetos únicos con Prob≥0.7
📊 Datos originales cargados: 5769 entradas
📊 Duplicados exactos eliminados: 2560
📊 Objetos únicos después de eliminar duplicados: 3209
📊 Objetos con Prob≥0.7: 2717

🎯 OBJETIVO: Encontrar parámetros para obtener ~1500 objetos
📊 Partiendo de 2717 objetos únicos con Prob≥0.7

🔍 Probando estrategia: A - Balanceada
   • Objetos obtenidos: 761
   • Tasa de retención: 28.0%
   • Errores medianos en muestra:
     F378: 0.543 mag
     F395: 0.543 mag
     F410: 0.543 mag
     F430: 0.443 mag
     F515: 0.221 mag
     F660: 0.067 mag
     F861: 0.124 mag

🔍 Probando estrategia: B - Permisiva
   • Objetos obtenidos: 1394
   • Tasa de retención: 51.3%
   • Errores medianos en muestra:
     F378: 0.543 mag
     F395: 0.543 mag
  